> **Archived research notebook.** Retained for reproducibility and comparisons; it may require the historical code/dependencies. Use the [current notebook catalog](https://github.com/vtavakkoli/TinyCeNN-LM/blob/main/notebooks/README.md) for new runs.


# SmolLM2 AM-CeNN Adaptive v4 — Colab

Version 4 is designed from the v3 evaluation rather than applying one attention approximation uniformly to all 30 layers.

**v4 changes**
- layer-adaptive exact windows: 32 / 48 / 64 / 96 tokens
- layer-adaptive AM-CeNN features: 256 / 320 / 384 / 512
- exact attention to the first 8 anchor tokens in every layer
- token-dependent per-head fusion gate with a tier-specific safety cap
- pretrained dense FFN remains unchanged
- Q/K/V/O use a very small protected learning rate
- teacher-aware anti-repetition objective
- held-out teacher/student CE, PPL, KL and repetition evaluation


In [ ]:
import importlib, pathlib, subprocess, sys

REPO_DIR = pathlib.Path('/content/TinyCeNN-LM')
if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'reset', '--hard', 'origin/main'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/vtavakkoli/TinyCeNN-LM.git', str(REPO_DIR)], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR)], check=True)
SRC_DIR = REPO_DIR / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
importlib.invalidate_caches()

import tinycenn_lm
commit = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', '--short', 'HEAD'], text=True).strip()
print('Repository ready:', REPO_DIR)
print('Commit:', commit)
print('tinycenn_lm:', tinycenn_lm.__file__)

In [ ]:
import os
from google.colab import userdata
from huggingface_hub import login

token = userdata.get('HF_TOKEN')
if not token:
    raise RuntimeError('Add a Hugging Face WRITE token to Colab Secrets as HF_TOKEN, then rerun this cell.')
os.environ['HF_TOKEN'] = token
login(token=token, add_to_git_credential=False)
print('Hugging Face login ready; mandatory TinyCeNN backup is enabled.')

## Configuration
The first run uses 500k distillation tokens so it fits comfortably in a normal Colab GPU session. If v4 beats v3, increase `FINAL_MAX_TOKENS` to 1–2M for the next run.

In [ ]:
from datetime import datetime, timezone

BASE_MODEL = 'HuggingFaceTB/SmolLM2-135M'
CONTEXT_LENGTH = 128
FINAL_MAX_TOKENS = 500_000
CALIBRATION_STEPS = 20
MAX_RUNTIME_MINUTES = 55

run_stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUTPUT_DIR = REPO_DIR / 'checkpoints' / f'smollm2-amcenn-adaptive-v4-{run_stamp}'
EVAL_JSON = OUTPUT_DIR / 'v4_evaluation.json'
print('Output:', OUTPUT_DIR)

## Train v4
The layer profile is based on the v3 calibration failures: layers 18 and 20 are treated as critical; 6, 7, 8, 14, 17, 23 and 26 as hard. The remaining layers use medium/easy profiles.

In [ ]:
cmd = [
    sys.executable, str(REPO_DIR / 'scripts' / 'train_smollm2_amcenn_v4.py'),
    '--base-model', BASE_MODEL,
    '--output-dir', str(OUTPUT_DIR),
    '--context-length', str(CONTEXT_LENGTH),
    '--anchor-tokens', '8',
    '--easy-window', '32', '--medium-window', '48', '--hard-window', '64', '--critical-window', '96',
    '--easy-feature-dim', '256', '--medium-feature-dim', '320', '--hard-feature-dim', '384', '--critical-feature-dim', '512',
    '--gate-init', '0.03',
    '--group-size', '1',
    '--calibration-steps', str(CALIBRATION_STEPS),
    '--calibration-lr', '8e-5',
    '--final-max-tokens', str(FINAL_MAX_TOKENS),
    '--memory-lr', '4e-5',
    '--qkvo-lr', '2e-6',
    '--ce-weight', '0.6', '--kl-weight', '1.2', '--hidden-weight', '0.30',
    '--repetition-weight', '0.10', '--recent-window', '8',
    '--final-eval-batches', '8',
    '--max-runtime-minutes', str(MAX_RUNTIME_MINUTES),
]
print(' '.join(cmd))
subprocess.run(cmd, check=True)

## Evaluate directly against SmolLM2-135M
This uses a separate 8,192-token held-out stream and the same four prompts used to diagnose v3.

In [ ]:
eval_cmd = [
    sys.executable, str(REPO_DIR / 'scripts' / 'evaluate_smollm2_amcenn_v4.py'),
    '--model-dir', str(OUTPUT_DIR),
    '--output', str(EVAL_JSON),
    '--eval-tokens', '8192',
    '--context-length', str(CONTEXT_LENGTH),
    '--max-new-tokens', '64',
]
print(' '.join(eval_cmd))
subprocess.run(eval_cmd, check=True)

## Quality dashboard
For reference, the previous v3 run had student PPL ≈265.17, KL ≈3.186 and roughly 0.44–0.53 3-gram repetition on the first three prompts. v4 should move all three downward.

In [ ]:
import json
import pandas as pd
from IPython.display import display

report = json.loads(EVAL_JSON.read_text())
summary = pd.DataFrame([{
    'teacher CE': report['teacher_ce'],
    'v4 CE': report['student_ce'],
    'CE gap': report['ce_gap'],
    'teacher PPL': report['teacher_perplexity'],
    'v4 PPL': report['student_perplexity'],
    'teacher↔v4 KL': report['student_teacher_kl'],
    'mean AM gate': report['attention_stats']['mean_global_gate'],
    'max observed AM gate': report['attention_stats']['max_observed_gate'],
}])
display(summary.round(4))

rows = []
for item in report['generations']:
    rows.append({
        'prompt': item['prompt'][:36],
        'teacher 3gram repeat': item['teacher_repeat_3gram_fraction'],
        'v4 3gram repeat': item['student_repeat_3gram_fraction'],
        'teacher recent-token repeat': item['teacher_recent_token_repeat_fraction'],
        'v4 recent-token repeat': item['student_recent_token_repeat_fraction'],
    })
display(pd.DataFrame(rows).round(4))

In [ ]:
for item in report['generations']:
    print('=' * 100)
    print('PROMPT:', item['prompt'])
    print('\nTEACHER:\n', item['teacher'])
    print('\nV4 STUDENT:\n', item['student'])
    print(f"\n3-gram repetition teacher={item['teacher_repeat_3gram_fraction']:.3f} v4={item['student_repeat_3gram_fraction']:.3f}")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(['Teacher', 'v3 baseline', 'v4'], [report['teacher_perplexity'], 265.1745, report['student_perplexity']])
ax.set_ylabel('Perplexity (lower is better)')
ax.set_title('SmolLM2 teacher vs AM-CeNN variants')
plt.show()